In [1]:
import pandas as pd·
from umap import UMAP
from sentence_transformers import SentenceTransformer
import pandas as pd
import numpy as np
import re
from collections import defaultdict
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestCentroid
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer
from sklearn.metrics import calinski_harabasz_score, silhouette_score
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import pandas as pd
from hdbscan import HDBSCAN
from bertopic import BERTopic
from sklearn.metrics.pairwise import cosine_similarity
import networkx as nx

### K-Means + UMAP Clustering Pipeline

This section implements the first part of our cognitively grounded clustering pipeline, as detailed in Section 3.1 of the project report. The goal is to automatically construct a high-level task taxonomy from fine-grained topic labels, supporting downstream modeling and interpretability.

**Key steps include:**

1. **Semantic Embedding**: Each unique topic label is embedded using the `all-MiniLM-L6-v2` model from SentenceTransformers, yielding 384-dimensional semantic vectors that capture conceptual similarity.
2. **Manifold-Based Reduction**: UMAP reduces the embedding dimensionality to 5 dimensions, preserving local neighborhood structure critical for cognitive task clustering.
3. **Clustering**: We apply K-Means with `K=30` to group semantically similar prompts into task clusters. Small, noisy clusters are reassigned to their nearest major cluster based on centroid distance to improve stability and interpretability.
4. **Hierarchical Labeling**: Each cluster is assigned a high-level label via a combination of semantic regex rules and TF-IDF keyword summarization, reflecting conceptual task categories such as `medical_diagnosis`, `programming_code_optimization`, and more.

In [2]:
#K-means + UMAP
gpt3_scores_df = pd.read_json(
    "_shared/data100-readonly/fa24_grad_project_data/nlp-chatbot-analysis_data/training-set/chatbot-arena-gpt3-scores.jsonl.gz", 
    lines=True,
    compression="gzip"
)
def flatten_list_columns(df, columns):
    for col in columns:
        df[col] = df[col].apply(
            lambda x: x if isinstance(x, list) else [x] if pd.notnull(x) else []
        )
        df[col] = df[col].apply(lambda x: ", ".join(map(str, x)) if x else None)
    return df

gpt3_scores_df = flatten_list_columns(
    gpt3_scores_df, 
    ["topic_modeling_1", "topic_modeling_2", "topic_modeling_3"]
)

all_topics = pd.concat([
    gpt3_scores_df["topic_modeling_1"],
    gpt3_scores_df["topic_modeling_2"],
    gpt3_scores_df["topic_modeling_3"]
]).dropna().unique()

embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
topic_embeddings = embedding_model.encode(all_topics, show_progress_bar=True)
print(f"Embedding dimension: {topic_embeddings.shape}")  # Output (N, 384)

print("\nRunning UMAP dimensionality reduction...")
# Maintain relatively high dimension
# umap_model = UMAP(n_components=15, random_state=42)

umap_model = UMAP(
    n_neighbors=100,
    n_components=5,
    min_dist=0.25,
    metric='cosine',
    random_state=42,
    densmap=False 
)
reduced_embeddings = umap_model.fit_transform(topic_embeddings)

scaler = StandardScaler()
reduced_embeddings_scaled = scaler.fit_transform(reduced_embeddings)

# Improved K-Means clustering
kmeans = KMeans(
    n_clusters=30,
    init='k-means++',
    n_init=50,
    max_iter=1000,
    tol=1e-6,
    algorithm='elkan',
    random_state=42
)
cluster_labels = kmeans.fit_predict(reduced_embeddings_scaled)

SEMANTIC_RULES = {
    # Programming & Development (from Google's TaskBench classification)
    r"\b(code|debug|algorithm|api|git)\b": "programming_code_generation",
    r"\b(optimiz|refactor|complexity)\b": "programming_code_optimization",
    r"\b(cyber|encrypt|vulnerability)\b": "programming_security",
    
    # Mathematics & Logic (inspired by BigBench framework)
    r"\b(equation|derivative|integral|matrix)\b": "mathematics_calculation",
    r"\b(proof|theorem|axiom|logical)\b": "mathematics_proving",
    r"\b(probability|statistic|distribution)\b": "mathematics_probability",
    
    # Medical & Health (based on BioBERT applications)
    r"\b(diagnos|symptom|pathology)\b": "medical_diagnosis",
    r"\b(drug|pharmaco|dosage)\b": "medical_pharmacy",
    r"\b(genetic|protein|dna)\b": "medical_genomics",
    
    # Law & Compliance (from LexGLUE benchmark)
    r"\b(contract|clause|agreement)\b": "legal_contract",
    r"\b(patent|copyright|intellectual)\b": "legal_ip",
    r"\b(compli|regulation|gdpr)\b": "legal_compliance",
    
    # Education (referenced from EdNet task taxonomy)
    r"\b(teach|curriculum|pedagogy)\b": "education_instruction",
    r"\b(exam|quiz|assessment)\b": "education_evaluation",
    r"\b(translate|grammar|vocabulary)\b": "language_translation",
    
    # Finance (based on FinBERT categories)
    r"\b(stock|portfolio|dividend)\b": "finance_investment",
    r"\b(loan|mortgage|interest)\b": "finance_banking",
    r"\b(tax|audit|compliance)\b": "finance_accounting",
    
    # Multimodal (based on Microsoft's NUWA framework)
    r"\b(image|vision|detection)\b": "multimodal_vision",
    r"\b(audio|speech|transcribe)\b": "multimodal_audio",
    r"\b(video|frame|streaming)\b": "multimodal_video"
}

def enhanced_label_generation(topics_list):
    """Generate hierarchical labels (domain + subdomain)"""
    combined_text = " ".join(topics_list).lower()
    
    # First layer: domain detection
    domain_hierarchy = {
        "programming": ["code_generation", "code_optimization", "security"],
        "mathematics": ["calculation", "proving", "probability"],
        "medical": ["diagnosis", "pharmacy", "genomics"],
        "legal": ["contract", "ip", "compliance"],
        "education": ["instruction", "evaluation", "translation"],
        "finance": ["investment", "banking", "accounting"],
        "multimodal": ["vision", "audio", "video"]
    }
    
    # Prioritize domain matching
    for domain in domain_hierarchy:
        if re.search(r"\b(" + "|".join(domain_hierarchy[domain]) + r")\b", combined_text):
            sub_pattern = "|".join([f"({s})" for s in domain_hierarchy[domain]])
            sub_match = re.search(sub_pattern, combined_text)
            sub_label = sub_match.group() if sub_match else "general"
            return f"{domain}_{sub_label}"
    
    # Fallback to TF-IDF keyword summarization
    tfidf = TfidfVectorizer(
        stop_words='english', 
        ngram_range=(1, 2),  # capture n-grams
        max_features=100
    )
    try:
        tfidf_matrix = tfidf.fit_transform(topics_list)
        feature_names = tfidf.get_feature_names_out()
        # Select top-5 keywords with filtering
        top_words = [
            word for word, _ in 
            sorted(zip(feature_names, np.asarray(tfidf_matrix.sum(axis=0).flatten())),
                   key=lambda x: -x[1])
            if len(word) > 4
        ][:5]
        return "_".join(top_words)
    except:
        return "cross_domain"
    
print("\nMerging small clusters...")
def merge_small_clusters(embeddings, labels, min_size_ratio=0.01):
    """Merge small clusters into nearest larger clusters"""
    min_size = int(len(labels) * min_size_ratio)
    unique_labels, counts = np.unique(labels, return_counts=True)
    
    # Split large and small clusters
    large_labels = unique_labels[counts >= min_size]
    small_labels = unique_labels[counts < min_size]
    
    # Return original if no valid large cluster exists
    if len(large_labels) == 0:
        return labels
    
    # Train nearest centroid classifier
    clf = NearestCentroid().fit(embeddings, labels)
    
    # Build label mapping
    label_map = {}
    for sl in small_labels:
        centroid = clf.centroids_[sl].reshape(1, -1)
        nearest_label = clf.predict(centroid)[0]
        label_map[sl] = nearest_label if nearest_label in large_labels else large_labels[0]
    
    return np.vectorize(lambda x: label_map.get(x, x))(labels)

processed_labels = merge_small_clusters(
    embeddings=reduced_embeddings_scaled,  
    labels=cluster_labels
)

# Construct topic -> cluster_id mapping
topic_to_cluster = {}
for topic_idx, label in enumerate(processed_labels):
    real_topic = all_topics[topic_idx]
    topic_to_cluster[real_topic] = label

print(f"Mapping completed: {len(topic_to_cluster)} topics assigned to Cluster IDs.")


from sklearn.metrics import silhouette_score, calinski_harabasz_score
from sklearn.metrics.pairwise import cosine_similarity
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from collections import Counter, defaultdict

# Generate statistics for each cluster: number of topics, example topics, high-level label
cluster_counter = Counter(processed_labels)
cluster_examples = defaultdict(list)

for topic, label in topic_to_cluster.items():
    cluster_examples[label].append(topic)

stats_data = []
for cluster_id in sorted(cluster_counter):
    topics_in_cluster = cluster_examples[cluster_id]
    high_level_label = enhanced_label_generation(topics_in_cluster)
    
    stats_data.append({
        "Cluster ID": cluster_id,
        "Topics Count": cluster_counter[cluster_id],
        "Example Topics": topics_in_cluster[:5],  # Only keep top 5 examples
        "High-Level Label": high_level_label
    })

# Construct summary DataFrame
stats_df = pd.DataFrame(stats_data)


### 1. Clustering Quality Evaluation ###
sil_score = silhouette_score(reduced_embeddings_scaled, processed_labels)
ch_score = calinski_harabasz_score(reduced_embeddings_scaled, processed_labels)

print(f"📐 Silhouette Score: {sil_score:.4f}")
print(f"📊 Calinski-Harabasz Index: {ch_score:.2f}")

### 2. Knowledge Graph Construction ###
kg = nx.Graph()
for _, row in stats_df.iterrows():
    kg.add_node(
        row["Cluster ID"],
        label=row["High-Level Label"],
        count=row["Topics Count"],
        examples=row["Example Topics"]
    )

# Compute centroids for each cluster
cluster_centroids = {}
for cluster_id in stats_df["Cluster ID"]:
    mask = (processed_labels == cluster_id)
    cluster_centroids[cluster_id] = reduced_embeddings_scaled[mask].mean(axis=0)

# Build similarity matrix between cluster centroids
cluster_ids = stats_df["Cluster ID"].tolist()
centroid_matrix = np.array([cluster_centroids[cid] for cid in cluster_ids])
similarity_matrix = cosine_similarity(centroid_matrix)

# Add high-similarity edges (top-5 or similarity > 0.9)
threshold = 0.9
for i, cid1 in enumerate(cluster_ids):
    for j, cid2 in enumerate(cluster_ids[i+1:], start=i+1):
        sim = similarity_matrix[i][j]
        if sim > threshold:
            kg.add_edge(cid1, cid2, weight=sim, label=f"Similarity: {sim:.2f}")

print(f"\n📎 Number of nodes in the knowledge graph: {kg.number_of_nodes()}, edges: {kg.number_of_edges()}")

### 3. Graph Structure Analysis ###
edge_weights = [d['weight'] for _, _, d in kg.edges(data=True)]
avg_edge_sim = np.mean(edge_weights) if edge_weights else 0.0
print(f"🔗 Average similarity of high-confidence edges: {avg_edge_sim:.4f}")

### 4. Similarity Report Generation ###
similarity_report = []
for cluster_id in stats_df["Cluster ID"]:
    sim_scores = similarity_matrix[cluster_ids.index(cluster_id)]
    top_indices = np.argsort(sim_scores)[-4:-1][::-1]

    related_clusters = [
        (cluster_ids[i], sim_scores[i])
        for i in top_indices if cluster_ids[i] != cluster_id
    ]

    similarity_report.append({
        "Cluster ID": cluster_id,
        "Main Label": kg.nodes[cluster_id]['label'],
        "Top Similar 1": f"{related_clusters[0][0]} ({related_clusters[0][1]:.2f})",
        "Top Similar 2": f"{related_clusters[1][0]} ({related_clusters[1][1]:.2f})",
        "Top Similar 3": f"{related_clusters[2][0]} ({related_clusters[2][1]:.2f})"
    })

similarity_df = pd.DataFrame(similarity_report)
print("\n🧭 Cluster Structural Similarity Report:")
print(similarity_df.to_markdown(index=False))




Batches:   0%|          | 0/681 [00:00<?, ?it/s]

Embedding dimension: (21767, 384)

Running UMAP dimensionality reduction...


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



Merging small clusters...
Mapping completed: 21767 topics assigned to Cluster IDs.
📐 Silhouette Score: 0.2977
📊 Calinski-Harabasz Index: 4074.96

📎 Number of nodes in the knowledge graph: 30, edges: 1
🔗 Average similarity of high-confidence edges: 0.9261

🧭 Cluster Structural Similarity Report:
|   Cluster ID | Main Label              | Top Similar 1   | Top Similar 2   | Top Similar 3   |
|-------------:|:------------------------|:----------------|:----------------|:----------------|
|            0 | mathematics_calculation | 17 (0.72)       | 10 (0.59)       | 27 (0.58)       |
|            1 | medical_diagnosis       | 15 (0.69)       | 19 (0.67)       | 22 (0.61)       |
|            2 | education_evaluation    | 6 (0.66)        | 3 (0.58)        | 14 (0.57)       |
|            3 | education_evaluation    | 12 (0.88)       | 28 (0.73)       | 2 (0.58)        |
|            4 | legal_ip                | 20 (0.66)       | 19 (0.57)       | 28 (0.56)       |
|            5 | mathema

### BERTopic + HDBSCAN + UMAP Clustering Pipeline

This section implements an alternative unsupervised task clustering pipeline that combines **semantic embedding**, **UMAP dimensionality reduction**, **density-based clustering via HDBSCAN**, and **interpretable topic modeling via BERTopic**.

**Key steps include:**

1. **Prompt Aggregation**: All topic labels from the dataset are preprocessed and combined to form the input prompt list.
2. **Embedding & Reduction**: We use a pretrained `all-mpnet-base-v2` model to embed the prompts, followed by UMAP to project embeddings into a 15-dimensional manifold space optimized for clustering.
3. **Clustering with BERTopic + HDBSCAN**: 
   - BERTopic integrates UMAP and HDBSCAN to produce high-quality, noise-aware clusters.
   - HDBSCAN identifies dense topic regions while BERTopic uses class-based TF-IDF (c-TF-IDF) to generate interpretable topic descriptors.
4. **Label Assignment**: Hierarchical labels are derived from domain-specific keyword rules and fallback TF-IDF-based summaries.
5. **Evaluation & Visualization**: We assess cluster quality using Silhouette Score and Calinski-Harabasz Index, and construct a knowledge graph to visualize inter-cluster semantic similarity.


In [3]:
# BerTopic+HDBSCAN

# Load the dataset
gpt3_scores_df = pd.read_json(
    "_shared/data100-readonly/fa24_grad_project_data/nlp-chatbot-analysis_data/training-set/chatbot-arena-gpt3-scores.jsonl.gz", 
    lines=True,
    compression="gzip"
)

# Convert list-based columns to comma-separated strings
def flatten_list_columns(df, columns):
    for col in columns:
        df[col] = df[col].apply(
            lambda x: x if isinstance(x, list) else [x] if pd.notnull(x) else []
        )
        df[col] = df[col].apply(lambda x: ", ".join(map(str, x)) if x else None)
    return df

# Preprocess the three topic modeling columns
gpt3_scores_df = flatten_list_columns(
    gpt3_scores_df, 
    ["topic_modeling_1", "topic_modeling_2", "topic_modeling_3"]
)

# Combine all topics into a single training set
all_topics = pd.concat([
    gpt3_scores_df["topic_modeling_1"],
    gpt3_scores_df["topic_modeling_2"],
    gpt3_scores_df["topic_modeling_3"]
]).dropna().unique()

all_prompts = all_topics
print(f"Total training prompts: {len(all_prompts)}")

#############################
# 2. BERTopic + HDBSCAN Training Pipeline
#############################

embedding_model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

umap_model = UMAP(
    n_neighbors=10,
    n_components=15,
    min_dist=0.01, 
    metric='cosine',
    random_state=42
)

hdbscan_model = HDBSCAN(
    min_cluster_size=15,
    cluster_selection_epsilon=0.1,
    cluster_selection_method='leaf',
    alpha=1.0,
    prediction_data=True
)

topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    n_gram_range=(1, 2),
    min_topic_size=15,
    calculate_probabilities=True,
    verbose=True
)

print("Step 4: BERTopic model initialized.")
topics, probs = topic_model.fit_transform(all_prompts)
print("Step 5: BERTopic model training completed.")

# Filter out low-confidence topic assignments
updated_topics = topic_model.reduce_outliers(
    all_prompts, 
    topics=topics, 
    probabilities=probs,
    threshold=0.01
)

# Recompute sentence embeddings
embeddings = embedding_model.encode(all_prompts, show_progress_bar=True)

# Project to lower dimensions using UMAP
reduced_embeddings = umap_model.transform(embeddings)

# Filter out noise-labeled samples (-1)
valid_mask = np.array(updated_topics) != -1
filtered_embeddings = reduced_embeddings[valid_mask]
filtered_labels = np.array(updated_topics)[valid_mask]

print(f"Remaining valid samples after filtering: {len(filtered_labels)}")

#############################
# 3. Hierarchical Label Generation
#############################
def enhanced_label_generation(topics_list):
    """Generate hierarchical labels based on domain + subdomain structure"""
    combined_text = " ".join(topics_list).lower()
    
    domain_hierarchy = {
        "programming": ["code_generation", "code_optimization", "security"],
        "mathematics": ["calculation", "proving", "probability"],
        "medical": ["diagnosis", "pharmacy", "genomics"],
        "legal": ["contract", "ip", "compliance"],
        "education": ["instruction", "evaluation", "translation"],
        "finance": ["investment", "banking", "accounting"],
        "multimodal": ["vision", "audio", "video"]
    }

    for domain in domain_hierarchy:
        if re.search(r"\b(" + "|".join(domain_hierarchy[domain]) + r")\b", combined_text):
            sub_pattern = "|".join([f"({s})" for s in domain_hierarchy[domain]])
            sub_match = re.search(sub_pattern, combined_text)
            sub_label = sub_match.group() if sub_match else "general"
            return f"{domain}_{sub_label}"
    
    tfidf = TfidfVectorizer(
        stop_words='english', 
        ngram_range=(1, 2),
        max_features=100
    )
    try:
        tfidf_matrix = tfidf.fit_transform(topics_list)
        feature_names = tfidf.get_feature_names_out()
        top_words = [
            word for word, _ in 
            sorted(zip(feature_names, np.asarray(tfidf_matrix.sum(axis=0).flatten())),
                   key=lambda x: -x[1])
            if len(word) > 4
        ][:5]
        return "_".join(top_words)
    except:
        return "cross_domain"

#############################
# 4. Cluster Statistics and Label Assignment
#############################
cluster_counter = Counter(filtered_labels)
cluster_examples = defaultdict(list)
for topic, label in zip(all_prompts[valid_mask], filtered_labels):
    cluster_examples[label].append(topic)

stats_data = []
for cluster_id in sorted(cluster_counter):
    topics_in_cluster = cluster_examples[cluster_id]
    high_level_label = enhanced_label_generation(topics_in_cluster)
    
    stats_data.append({
        "Cluster ID": cluster_id,
        "Topics Count": cluster_counter[cluster_id],
        "Example Topics": topics_in_cluster[:5],
        "High-Level Label": high_level_label
    })

stats_df = pd.DataFrame(stats_data)
print(f"Mapping complete. Generated statistics for {len(stats_df)} topic clusters.")

# Clustering Metrics: Silhouette Score and Calinski-Harabasz Index
sil_score = silhouette_score(filtered_embeddings, filtered_labels)
ch_score = calinski_harabasz_score(filtered_embeddings, filtered_labels)
print(f"\n📐 Silhouette Score: {sil_score:.4f}")
print(f"📊 Calinski-Harabasz Index: {ch_score:.2f}")

# Build Knowledge Graph
kg = nx.Graph()
for _, row in stats_df.iterrows():
    kg.add_node(
        row["Cluster ID"],
        label=row["High-Level Label"],
        count=row["Topics Count"],
        examples=row["Example Topics"]
    )

print(f"\nKnowledge graph contains {kg.number_of_nodes()} nodes")

# Compute centroid for each cluster
cluster_centroids = {}
for cluster_id in stats_df["Cluster ID"]:
    mask = (filtered_labels == cluster_id)
    cluster_centroids[cluster_id] = filtered_embeddings[mask].mean(axis=0)

# Generate similarity matrix between cluster centroids
cluster_ids = stats_df["Cluster ID"].tolist()
centroid_matrix = np.array([cluster_centroids[cid] for cid in cluster_ids])
similarity_matrix = cosine_similarity(centroid_matrix)

# Add high similarity edges (adjustable threshold)
threshold = 0.9
for i, cid1 in enumerate(cluster_ids):
    for j, cid2 in enumerate(cluster_ids[i+1:], start=i+1):
        sim = similarity_matrix[i][j]
        if sim > threshold:
            kg.add_edge(cid1, cid2, weight=sim, label=f"Similarity: {sim:.2f}")

print(f"Knowledge graph contains {kg.number_of_edges()} high-similarity edges")

# Generate similarity analysis report
similarity_report = []
for cluster_id in stats_df["Cluster ID"]:
    sim_scores = similarity_matrix[cluster_ids.index(cluster_id)]
    # Get top 3 most similar clusters (excluding self)
    top_indices = np.argsort(sim_scores)[-4:-1][::-1]
    related_clusters = [
        (cluster_ids[i], sim_scores[i])
        for i in top_indices 
        if cluster_ids[i] != cluster_id
    ]
    similarity_report.append({
        "Cluster ID": cluster_id,
        "Main Label": kg.nodes[cluster_id]['label'],
        "Top Similar 1": f"{related_clusters[0][0]} ({related_clusters[0][1]:.2f})",
        "Top Similar 2": f"{related_clusters[1][0]} ({related_clusters[1][1]:.2f})",
        "Top Similar 3": f"{related_clusters[2][0]} ({related_clusters[2][1]:.2f})"
    })

similarity_df = pd.DataFrame(similarity_report)
print("\n🧭 Cluster Similarity Structure Report:")
print(similarity_df.to_markdown(index=False))



Total training prompts: 21767


2025-04-24 19:26:50,336 - BERTopic - Embedding - Transforming documents to embeddings.


Step 4: BERTopic model initialized.


Batches:   0%|          | 0/681 [00:00<?, ?it/s]

2025-04-24 19:27:00,921 - BERTopic - Embedding - Completed ✓
2025-04-24 19:27:00,921 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-04-24 19:27:12,389 - BERTopic - Dimensionality - Completed ✓
2025-04-24 19:27:12,390 - BERTopic - Cluster - Start clustering the reduced embeddings
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current p

Step 5: BERTopic model training completed.


100%|██████████| 7/7 [00:00<00:00, 121.68it/s]


Batches:   0%|          | 0/681 [00:00<?, ?it/s]

Remaining valid samples after filtering: 21534
Mapping complete. Generated statistics for 374 topic clusters.

📐 Silhouette Score: 0.0588
📊 Calinski-Harabasz Index: 249.74

Knowledge graph contains 374 nodes
Knowledge graph contains 69296 high-similarity edges

🧭 Cluster Similarity Structure Report:
|   Cluster ID | Main Label              | Top Similar 1   | Top Similar 2   | Top Similar 3   |
|-------------:|:------------------------|:----------------|:----------------|:----------------|
|            0 | education_evaluation    | 115 (0.99)      | 32 (0.99)       | 47 (0.99)       |
|            1 | education_evaluation    | 221 (1.00)      | 131 (1.00)      | 336 (1.00)      |
|            2 | education_evaluation    | 237 (1.00)      | 364 (1.00)      | 108 (1.00)      |
|            3 | mathematics_calculation | 352 (1.00)      | 124 (1.00)      | 94 (1.00)       |
|            4 | mathematics_calculation | 67 (1.00)       | 349 (1.00)      | 64 (1.00)       |
|            5 | edu